# Tratamento — Procedures by Country (Bronze -> Silver)

Este notebook lê os Parquet da camada **Bronze** (extração tabular bruta dos PDFs ISAPS, sem nenhuma limpeza) e aplica as transformações necessárias para a camada **Silver** — o dado confiável, consistente e pronto para ser consumido por análises e dashboards.

Cada transformação é uma etapa separada, na ordem abaixo, e cada uma tem sua própria célula markdown explicando **o que** é feito e **por quê** (a EDA em [exploracao_bronze_isaps.ipynb](exploracao_bronze_isaps.ipynb) é a origem de todos os problemas identificados aqui):

1. **Deduplicação** — remove linhas totalmente repetidas.
2. **Remoção de nulos em `quantidade`** — descarta procedimentos não reportados.
3. **Normalização de país** — unifica grafias diferentes do mesmo país entre relatórios de anos distintos.
4. **Tradução PT-BR** — traduz `categoria`/`procedimento` via dicionário de mapeamento, preservando o termo original em inglês.
5. **Validação de schema (`pandera`)** — garante o contrato de dados final da Silver antes da gravação.

Ao final, o resultado é gravado em Parquet (um arquivo por PDF de origem) localmente em `dados_processados/silver/procedures_by_country/` e no MinIO, bucket `silver`.

## 1. Imports e configuração

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from minio import Minio
from pandera.pandas import Column, Check, DataFrameSchema

In [2]:
load_dotenv(Path.cwd().parent / ".env")

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "localhost:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")

BUCKET_BRONZE = os.getenv("BUCKET_BRONZE", "bronze")
BRONZE_PREFIX = "procedures_by_country/"

BUCKET_SILVER = os.getenv("BUCKET_SILVER", "silver")
SILVER_PREFIX = "procedures_by_country/"

SILVER_DIR = Path.cwd().parent / "dados_processados" / "silver" / "procedures_by_country"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

client = Minio(MINIO_ENDPOINT, access_key=MINIO_ACCESS_KEY, secret_key=MINIO_SECRET_KEY, secure=False)
if not client.bucket_exists(BUCKET_SILVER):
    client.make_bucket(BUCKET_SILVER)
    print(f"Bucket '{BUCKET_SILVER}' criado.")

print(f"MinIO: {MINIO_ENDPOINT} | bucket bronze: {BUCKET_BRONZE} | bucket silver: {BUCKET_SILVER}")
print(f"Saida parquet local: {SILVER_DIR}")

MinIO: localhost:9000 | bucket bronze: bronze | bucket silver: silver
Saida parquet local: C:\Projeto_AI\dados_processados\silver\procedures_by_country


## 2. Carregamento da camada Bronze

Lê todos os Parquet gravados pela extração Bronze ([extracao_bronze_isaps.ipynb](extracao_bronze_isaps.ipynb)) direto do MinIO (bucket `bronze`) e consolida num único DataFrame. As colunas `arquivo_origem` e `ano_referencia` já vêm da Bronze e funcionam como metadado de rastreabilidade — toda linha da Silver poderá ser conferida contra o PDF e o ano que a originou.

In [3]:
objects = list(client.list_objects(BUCKET_BRONZE, prefix=BRONZE_PREFIX, recursive=True))
print(f"{len(objects)} arquivo(s) encontrados na Bronze:")
for obj in objects:
    print(" -", obj.object_name)

dfs_bronze = []
for obj in objects:
    data = client.get_object(BUCKET_BRONZE, obj.object_name).read()
    dfs_bronze.append(pd.read_parquet(pd.io.common.BytesIO(data)))

df = pd.concat(dfs_bronze, ignore_index=True)
print(f"\nShape consolidado (Bronze): {df.shape}")

7 arquivo(s) encontrados na Bronze:
 - procedures_by_country/2018_isaps_global_survey_results_2018_1.parquet
 - procedures_by_country/2019_global_survey_full_report_2019_english.parquet
 - procedures_by_country/2020_isaps_global_survey_2020.parquet
 - procedures_by_country/2021_isaps_global_survey_2021.parquet
 - procedures_by_country/2022_isaps_global_survey_2022.parquet
 - procedures_by_country/2023_isaps_global_survey_2023.parquet
 - procedures_by_country/2024_isaps_global_survey_2024.parquet

Shape consolidado (Bronze): (5384, 7)


## 3. Etapa 1 — Deduplicação

**O que é feito:** remove linhas em que **todas** as colunas (`arquivo_origem`, `ano_referencia`, `tipo_procedimento`, `categoria`, `procedimento`, `pais`, `quantidade`) são idênticas a outra linha.

**Por que é necessária, mesmo a EDA não tendo encontrado duplicatas hoje:**

- A extração da Bronze usa `pdfplumber.extract_tables()`, que em alguns layouts de PDF pode devolver a **mesma tabela detectada mais de uma vez** (por exemplo, quando uma tabela grande é fragmentada em blocos que se sobrepõem na página) ou capturar a mesma linha em duas passagens diferentes. Isso é uma característica conhecida da extração de tabelas em PDF, não um bug específico deste código — e pode variar de relatório para relatório sem aviso.
- Se o notebook de extração Bronze for reexecutado de forma incremental (ex.: reprocessar só o PDF mais novo, mas sem limpar o bucket antes) ou se um mesmo PDF for reenviado por engano à camada raw, o resultado é o **mesmo dado aparecendo duplicado** quando os Parquet forem consolidados aqui.
- Uma linha duplicada não gera um erro visível — ela **infla silenciosamente as agregações**. Uma soma de `quantidade` por país, por exemplo, contaria aquele procedimento duas vezes, distorcendo o resultado sem que nada "quebre". Esse tipo de erro é caro de detectar depois, porque o número errado parece plausível.
- Rodar `drop_duplicates()` quando não há nada para remover **não tem custo** nem efeito colateral. Por isso a deduplicação é sempre aplicada na Silver como uma proteção estrutural, independente de o dado de entrada estar "limpo" hoje: ela garante que a Silver continua correta mesmo se um problema de extração aparecer amanhã, sem que ninguém precise perceber e corrigir manualmente.

In [4]:
antes = len(df)
df = df.drop_duplicates()
depois = len(df)
print(f"Linhas antes: {antes} | depois: {depois} | duplicadas removidas: {antes - depois}")

Linhas antes: 5384 | depois: 5384 | duplicadas removidas: 0


## 4. Etapa 2 — Remoção de nulos em `quantidade`

**O que é feito:** remove linhas onde `quantidade` é nula.

**Por que:** um valor nulo em `quantidade` significa que aquele procedimento **não foi reportado** por aquele país naquele ano — o relatório simplesmente não tem esse dado, o que é diferente de "zero procedimentos realizados". Manter o `NaN` traria dois problemas: (1) não seria possível validar a coluna como `float` não-nula no schema final, e (2) qualquer agregação (soma, média) feita sobre a coluna teria que decidir implicitamente como tratar o nulo — o `pandas` já ignora `NaN` em `sum()`/`mean()` por padrão, mas isso mascara a ausência de dado em vez de deixá-la explícita. Como a linha em si não carrega nenhuma informação utilizável sem a quantidade, o mais seguro é descartá-la aqui, de forma explícita e documentada, em vez de deixá-la implicitamente ignorada mais adiante.

`categoria` pode continuar nula legitimamente — linhas de totalização (ex.: `Total Procedures`) não pertencem a nenhuma categoria específica, e isso é um dado real, não uma ausência de informação.

In [5]:
antes = len(df)
df = df.dropna(subset=["quantidade"])
depois = len(df)
print(f"Linhas antes: {antes} | depois: {depois} | nulos removidos: {antes - depois}")

Linhas antes: 5384 | depois: 5382 | nulos removidos: 2


## 5. Etapa 3 — Normalização de país

**O que é feito:** unifica variantes de grafia do mesmo país que aparecem em relatórios de anos diferentes, mapeando-as para um único nome canônico.

**Por que:** a EDA (seção 6.1 de [exploracao_bronze_isaps.ipynb](exploracao_bronze_isaps.ipynb)) mostrou que o ISAPS não usa uma grafia consistente de país entre edições do relatório:

| Variantes encontradas | Linhas (cada variante) |
|---|---|
| `USA` / `US` | 161 / 133 |
| `UK` / `UNITED KINGDOM` | 91 / 82 |
| `TURKEY` / `TURKIYE` | 164 / 91 |

Sem essa normalização, qualquer agregação "por país" (ex.: total de procedimentos nos EUA ao longo dos anos) fica **fisicamente errada**: o mesmo país aparece como duas entidades diferentes, e cada uma concentra só uma fração dos anos — o total real fica dividido e nenhuma das duas linhas mostra o número correto. Isso não é um erro visível (nenhum país "some"), é uma distorção silenciosa que só aparece se alguém comparar manualmente as duas grafias.

O nome canônico escolhido para cada grupo foi o de **maior frequência** nos dados (mais edições do relatório usam essa grafia), para minimizar o número de linhas remapeadas e manter o nome mais "familiar" no histórico:

In [6]:
PAIS_NORMALIZACAO = {
    "US": "USA",
    "UNITED KINGDOM": "UK",
    "TURKIYE": "TURKEY",
}

paises_antes = df["pais"].nunique()
linhas_afetadas = df["pais"].isin(PAIS_NORMALIZACAO).sum()

df["pais"] = df["pais"].replace(PAIS_NORMALIZACAO)

paises_depois = df["pais"].nunique()
print(f"Paises distintos antes: {paises_antes} | depois: {paises_depois}")
print(f"Linhas remapeadas: {linhas_afetadas}")

Paises distintos antes: 41 | depois: 38
Linhas remapeadas: 306


**Reconferência de duplicatas:** normalizar o país pode, em teoria, fazer duas linhas que eram diferentes (por causa da grafia) colidirem numa única combinação `pais`/`procedimento`/`ano` idêntica. Isso só aconteceria se a mesma linha existisse duas vezes com grafias diferentes no mesmo arquivo — não é o caso aqui, mas a checagem abaixo confirma isso explicitamente em vez de assumir.

In [7]:
antes = len(df)
df = df.drop_duplicates()
depois = len(df)
print(f"Duplicatas geradas pela normalizacao de pais: {antes - depois}")

Duplicatas geradas pela normalizacao de pais: 0


## 6. Etapa 4 — Tradução PT-BR (`categoria` e `procedimento`)

**O que é feito:** traduz `categoria` e `procedimento` para português via **dicionário de mapeamento** (`{"Brow Lift": "Lifting de Sobrancelha", ...}`), gravando o resultado em duas colunas novas (`categoria_pt`, `procedimento_pt`) — **mantendo as colunas originais em inglês** (`categoria`, `procedimento`) intactas.

**Por que um dicionário, e por que manter o original:**

- Um dicionário fixo é **auditável**: qualquer pessoa consegue ver exatamente qual termo em inglês virou qual termo em português, célula por célula, sem "mágica" de tradução automática que poderia errar termos técnicos de procedimentos estéticos (ex.: traduzir errado teria impacto direto na credibilidade da Silver).
- Manter a coluna original em inglês preserva a **rastreabilidade com a fonte** (o PDF usa os termos em inglês) e permite conferir/corrigir a tradução a qualquer momento sem precisar reprocessar a Bronze.
- Se um relatório futuro trouxer um termo novo que ainda não está no dicionário, o pipeline **não falha silenciosamente** nem inventa uma tradução: a célula de validação abaixo lista qualquer valor sem mapeamento, para que o dicionário seja atualizado antes da tradução ser aplicada.

Note que variantes de grafia do **mesmo** procedimento entre anos diferentes (ex.: `Face Lift` e `Facelift`; `Non-Surgical Fat Reduction` e `Nonsurgical Fat Reduction`) foram traduzidas para o **mesmo termo em português**, o que já ajuda a leitura — mas as linhas **não foram fundidas**: cada uma continua sendo uma linha própria, com sua `procedimento` original em inglês preservada. Unificar essas variantes em um único identificador de procedimento (o que mudaria a granularidade dos dados) é uma decisão maior, fora do escopo deste tratamento, e fica registrada como ponto em aberto na conclusão.

In [8]:
CATEGORIA_PT = {
    "BODY & EXTREMITIES": "Corpo e Extremidades",
    "BREAST": "Mama",
    "FACE & HEAD": "Face e Cabeça",
    "FACIAL REJUVENATION": "Rejuvenescimento Facial",
    "INJECTABLES": "Injetáveis",
    "OTHER": "Outros",
}

PROCEDIMENTO_PT = {
    "Abdominoplasty": "Abdominoplastia",
    "Botulinum Toxin": "Toxina Botulínica",
    "Brachioplasty": "Braquioplastia",
    "Breast Augmentation": "Aumento de Mama",
    "Breast Implant Removal": "Remoção de Implante Mamário",
    "Breast Lift": "Lifting de Mama",
    "Breast Reduction": "Redução de Mama",
    "Brow Lift": "Lifting de Sobrancelha",
    "Buccal Fat Removal": "Bichectomia",
    "Buttock Augmentation": "Aumento de Glúteo",
    "Buttock Augmentation (implants and fat transfer)": "Aumento de Glúteo (Implantes e Enxerto de Gordura)",
    "Buttock Augmentation – Implants and Fat Transfer": "Aumento de Glúteo – Implantes e Enxerto de Gordura",
    "Buttock Lift": "Lifting de Glúteo",
    "Calcium Hydroxyapatite": "Hidroxiapatita de Cálcio",
    "Calcium Hydroxylapatite": "Hidroxiapatita de Cálcio",
    "Cellulite Treatment": "Tratamento de Celulite",
    "Chemical Peel": "Peeling Químico",
    "Dimple Creation": "Dimpleplastia (Criação de Covinhas)",
    "Ear Surgery": "Otoplastia",
    "Eyelid Surgery": "Blefaroplastia",
    "Face Lift": "Lifting Facial",
    "Facelift": "Lifting Facial",
    "Facial Bone Contouring": "Contorno Ósseo Facial",
    "Fat Grafting (face)": "Enxerto de Gordura na Face",
    "Fat Grafting – Face": "Enxerto de Gordura na Face",
    "Fat Grafting-face": "Enxerto de Gordura na Face",
    "Fat Reduction": "Redução de Gordura",
    "Full Field Ablative": "Ablativo de Campo Total",
    "Gynecomastia": "Ginecomastia",
    "Hair Removal": "Depilação",
    "Hand Rejuvenation with Fat Grafting": "Rejuvenescimento das Mãos com Enxerto de Gordura",
    "Hyaluronic Acid": "Ácido Hialurônico",
    "Inverted Nipple Correction": "Correção de Mamilo Invertido",
    "Labiaplasty": "Ninfoplastia",
    "Labiaplasty (excluding vaginal rejuvenation)": "Ninfoplastia (excluindo rejuvenescimento vaginal)",
    "Lip Enhancement Perioral Procedure": "Preenchimento Labial / Procedimento Perioral",
    "Lip Enhancement/ Perioral Procedure": "Preenchimento Labial / Procedimento Perioral",
    "Lip Enhancement/Perioral Procedure": "Preenchimento Labial / Procedimento Perioral",
    "Liposuction": "Lipoaspiração",
    "Lower Body Lift": "Lifting Corporal Inferior",
    "Micro-Ablative Resurfacing": "Resurfacing Micro-Ablativo",
    "Neck Lift": "Lifting de Pescoço",
    "Non-Surgical Fat Reduction": "Redução de Gordura Não Cirúrgica",
    "Non-Surgical Skin Tightening": "Firmeza de Pele Não Cirúrgica",
    "Nonsurgical Fat Reduction": "Redução de Gordura Não Cirúrgica",
    "Other Outer Genital Surgery": "Outras Cirurgias Genitais Externas",
    "Photo Rejuvenation": "Fotorejuvenescimento",
    "Poly-L-Lactic Acid": "Ácido Poli-L-Lático",
    "Rhinoplasty": "Rinoplastia",
    "Scar Revision": "Revisão de Cicatriz",
    "Surgical Vaginal Rejuvenation": "Rejuvenescimento Vaginal Cirúrgico",
    "TOTAL PROCEDURES": "TOTAL DE PROCEDIMENTOS",
    "Tattoo Removal": "Remoção de Tatuagem",
    "Thigh Lift": "Lifting de Coxa",
    "Total Body & Extremities": "Total de Corpo e Extremidades",
    "Total Body & Extremities Procedures": "Total de Procedimentos de Corpo e Extremidades",
    "Total Breast": "Total de Mama",
    "Total Breast Procedures": "Total de Procedimentos de Mama",
    "Total Face & Head": "Total de Face e Cabeça",
    "Total Face & Head Procedures": "Total de Procedimentos de Face e Cabeça",
    "Total Facial Rejuvenation": "Total de Rejuvenescimento Facial",
    "Total Facial Rejuvenation Procedures": "Total de Procedimentos de Rejuvenescimento Facial",
    "Total Injectables": "Total de Injetáveis",
    "Total Injectables Procedures": "Total de Procedimentos Injetáveis",
    "Total Non-Surgical": "Total Não Cirúrgico",
    "Total Non-Surgical Procedures": "Total de Procedimentos Não Cirúrgicos",
    "Total Nonsurgical Procedures": "Total de Procedimentos Não Cirúrgicos",
    "Total Other": "Total de Outros",
    "Total Other Procedures": "Total de Outros Procedimentos",
    "Total Procedures": "Total de Procedimentos",
    "Total SURGICAL & NON-SURGICAL Procedures": "Total de Procedimentos Cirúrgicos e Não Cirúrgicos",
    "Total Surgical Procedures": "Total de Procedimentos Cirúrgicos",
    "Upper Arm Lift": "Lifting de Braço Superior",
    "Upper Body Lift": "Lifting Corporal Superior",
    "Vaginal Rejuvenation": "Rejuvenescimento Vaginal",
}

**Validação do dicionário antes de aplicar:** confere se todo valor distinto de `categoria` (não nulo) e `procedimento` presente nos dados tem uma tradução no dicionário. Se faltar algum, o pipeline avisa explicitamente em vez de gravar um valor traduzido incompleto (`NaN`) sem ninguém perceber.

In [9]:
categorias_sem_mapeamento = sorted(set(df["categoria"].dropna().unique()) - set(CATEGORIA_PT))
procedimentos_sem_mapeamento = sorted(set(df["procedimento"].unique()) - set(PROCEDIMENTO_PT))

if categorias_sem_mapeamento or procedimentos_sem_mapeamento:
    raise ValueError(
        f"Dicionario de traducao incompleto. "
        f"Categorias sem mapeamento: {categorias_sem_mapeamento} | "
        f"Procedimentos sem mapeamento: {procedimentos_sem_mapeamento}"
    )

print("Dicionario cobre 100% dos valores distintos de categoria e procedimento.")

Dicionario cobre 100% dos valores distintos de categoria e procedimento.


In [10]:
df["categoria_pt"] = df["categoria"].map(CATEGORIA_PT)
df["procedimento_pt"] = df["procedimento"].map(PROCEDIMENTO_PT)

df[["categoria", "categoria_pt", "procedimento", "procedimento_pt"]].drop_duplicates().sort_values(
    ["categoria", "procedimento"]
).head(15)

,categoria,categoria_pt,procedimento,procedimento_pt
140,BODY & EXTREMITIES,Corpo e Extremidades,Abdominoplasty,Abdominoplastia
3976,BODY & EXTREMITIES,Corpo e Extremidades,Brachioplasty,Braquioplastia
150,BODY & EXTREMITIES,Corpo e Extremidades,Buttock Augmentation,Aumento de Glúteo
1254,BODY & EXTREMITIES,Corpo e Extremidades,Buttock Augmentation (implants and fat transfer),Aumento de Glúteo (Implantes e Enxerto de Gord...
1828,BODY & EXTREMITIES,Corpo e Extremidades,Buttock Augmentation – Implants and Fat Transfer,Aumento de Glúteo – Implantes e Enxerto de Gor...
160,BODY & EXTREMITIES,Corpo e Extremidades,Buttock Lift,Lifting de Glúteo
4000,BODY & EXTREMITIES,Corpo e Extremidades,Hand Rejuvenation with Fat Grafting,Rejuvenescimento das Mãos com Enxerto de Gordura
210,BODY & EXTREMITIES,Corpo e Extremidades,Labiaplasty,Ninfoplastia
1338,BODY & EXTREMITIES,Corpo e Extremidades,Labiaplasty (excluding vaginal rejuvenation),Ninfoplastia (excluindo rejuvenescimento vaginal)
170,BODY & EXTREMITIES,Corpo e Extremidades,Liposuction,Lipoaspiração


## 7. Etapa 5 — Validação de schema (`pandera`)

**O que é feito:** define o contrato de dados final da Silver com `pandera.DataFrameSchema` — tipos, obrigatoriedade de não-nulo, e domínios válidos (`tipo_procedimento` restrito a `{CIRURGICO, NAO-CIRURGICO}`, `quantidade >= 0`) — e valida o DataFrame tratado contra ele.

**Por que validar, se as etapas anteriores já deveriam garantir isso:** a validação de schema não confia "de olho" que as etapas 1 a 4 funcionaram — ela **prova** isso de forma automática e reprodutível. Se uma mudança futura no código (por exemplo, alguém editar o dicionário de tradução ou ajustar a normalização de país) introduzir um valor inesperado — um `tipo_procedimento` fora do domínio esperado, uma `quantidade` negativa, um nulo que não deveria existir mais —, o `schema.validate()` **lança uma exceção imediatamente**, no notebook, antes do dado errado ser gravado em Parquet e consumido por outra ferramenta. Sem essa etapa, um erro de tratamento só seria descoberto quando alguém notasse um número estranho num dashboard, bem mais tarde e bem mais difícil de rastrear até a causa.

`strict=True` também garante que nenhuma coluna extra ou renomeada passe despercebida — o schema é a definição explícita e única de como é uma linha válida da Silver.

In [11]:
schema = DataFrameSchema(
    {
        "arquivo_origem": Column(str, nullable=False),
        "ano_referencia": Column(int, Check.in_range(2000, 2100), nullable=False),
        "tipo_procedimento": Column(str, Check.isin(["CIRURGICO", "NAO-CIRURGICO"]), nullable=False),
        "categoria": Column(str, nullable=True),
        "categoria_pt": Column(str, nullable=True),
        "procedimento": Column(str, Check.str_length(min_value=1), nullable=False),
        "procedimento_pt": Column(str, Check.str_length(min_value=1), nullable=False),
        "pais": Column(str, Check.str_length(min_value=1), nullable=False),
        "quantidade": Column(float, Check.ge(0), nullable=False),
    },
    coerce=True,
    strict=True,
)

COLS_FINAIS = [
    "arquivo_origem", "ano_referencia", "tipo_procedimento",
    "categoria", "categoria_pt", "procedimento", "procedimento_pt",
    "pais", "quantidade",
]

df = df[COLS_FINAIS].reset_index(drop=True)
schema.validate(df)  # lanca excecao se o schema nao for respeitado
print(f"Schema valido. Shape final (Silver): {df.shape}")

Schema valido. Shape final (Silver): (5382, 9)


## 8. Gravação em Parquet (camada Silver)

Um arquivo por PDF de origem (mesma convenção da Bronze), gravado localmente em `dados_processados/silver/procedures_by_country/` e enviado ao MinIO no bucket `silver`.

In [12]:
arquivos_gerados = []
for fname, grupo in df.groupby("arquivo_origem"):
    ano = grupo["ano_referencia"].iloc[0]
    slug = re.sub(r"[^a-z0-9]+", "_", Path(fname).stem.lower()).strip("_")
    out_name = f"{ano}_{slug}.parquet"
    out_path = SILVER_DIR / out_name

    grupo = grupo.reset_index(drop=True)
    grupo.to_parquet(out_path, engine="pyarrow", index=False)
    arquivos_gerados.append(out_path)

    object_name = f"{SILVER_PREFIX}{out_name}"
    client.fput_object(BUCKET_SILVER, object_name, str(out_path))
    print(f"Gravado: {out_path} ({len(grupo)} linhas) -> s3://{BUCKET_SILVER}/{object_name}")

Gravado: C:\Projeto_AI\dados_processados\silver\procedures_by_country\2019_global_survey_full_report_2019_english.parquet (640 linhas) -> s3://silver/procedures_by_country/2019_global_survey_full_report_2019_english.parquet


Gravado: C:\Projeto_AI\dados_processados\silver\procedures_by_country\2024_isaps_global_survey_2024.parquet (1568 linhas) -> s3://silver/procedures_by_country/2024_isaps_global_survey_2024.parquet
Gravado: C:\Projeto_AI\dados_processados\silver\procedures_by_country\2018_isaps_global_survey_results_2018_1.parquet (390 linhas) -> s3://silver/procedures_by_country/2018_isaps_global_survey_results_2018_1.parquet


Gravado: C:\Projeto_AI\dados_processados\silver\procedures_by_country\2020_isaps_global_survey_2020.parquet (560 linhas) -> s3://silver/procedures_by_country/2020_isaps_global_survey_2020.parquet


Gravado: C:\Projeto_AI\dados_processados\silver\procedures_by_country\2021_isaps_global_survey_2021.parquet (587 linhas) -> s3://silver/procedures_by_country/2021_isaps_global_survey_2021.parquet


Gravado: C:\Projeto_AI\dados_processados\silver\procedures_by_country\2022_isaps_global_survey_2022.parquet (671 linhas) -> s3://silver/procedures_by_country/2022_isaps_global_survey_2022.parquet
Gravado: C:\Projeto_AI\dados_processados\silver\procedures_by_country\2023_isaps_global_survey_2023.parquet (966 linhas) -> s3://silver/procedures_by_country/2023_isaps_global_survey_2023.parquet


## 9. Conferência final

In [13]:
consolidado = pd.concat([pd.read_parquet(p) for p in arquivos_gerados], ignore_index=True)
print(f"Total consolidado: {len(consolidado)} linhas, {consolidado['arquivo_origem'].nunique()} arquivos, "
      f"{consolidado['pais'].nunique()} paises (ja normalizados), anos {sorted(consolidado['ano_referencia'].unique())}")
consolidado.sample(10, random_state=42)

Total consolidado: 5382 linhas, 7 arquivos, 38 paises (ja normalizados), anos [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


,arquivo_origem,ano_referencia,tipo_procedimento,categoria,categoria_pt,procedimento,procedimento_pt,pais,quantidade
3271,isaps-global-survey_2021.pdf,2021,CIRURGICO,FACE & HEAD,Face e Cabeça,Rhinoplasty,Rinoplastia,JAPAN,20128.0
907,isaps-global-survey-2024.pdf,2024,CIRURGICO,BODY & EXTREMITIES,Corpo e Extremidades,Total Surgical Procedures,Total de Procedimentos Cirúrgicos,BRAZIL,2354513.0
4579,isaps-global-survey_2023.pdf,2023,CIRURGICO,BREAST,Mama,Breast Implant Removal,Remoção de Implante Mamário,IRAN,1665.0
3463,isaps-global-survey_2021.pdf,2021,CIRURGICO,BODY & EXTREMITIES,Corpo e Extremidades,Thigh Lift,Lifting de Coxa,ROMANIA,259.0
319,global-survey-full-report-2019-english.pdf,2019,CIRURGICO,BODY & EXTREMITIES,Corpo e Extremidades,Lower Body Lift,Lifting Corporal Inferior,THAILAND,601.0
1717,isaps-global-survey-2024.pdf,2024,CIRURGICO,BODY & EXTREMITIES,Corpo e Extremidades,Other Outer Genital Surgery,Outras Cirurgias Genitais Externas,UK,510.0
3616,isaps-global-survey_2021.pdf,2021,NAO-CIRURGICO,FACIAL REJUVENATION,Rejuvenescimento Facial,Chemical Peel,Peeling Químico,GREECE,6934.0
2093,isaps-global-survey-2024.pdf,2024,NAO-CIRURGICO,OTHER,Outros,Hair Removal,Depilação,SOUTH AFRICA,0.0
5212,isaps-global-survey_2023.pdf,2023,NAO-CIRURGICO,OTHER,Outros,Total Non-Surgical Procedures,Total de Procedimentos Não Cirúrgicos,INDIA,496931.0
1593,isaps-global-survey-2024.pdf,2024,CIRURGICO,FACE & HEAD,Face e Cabeça,Brow Lift,Lifting de Sobrancelha,UK,1531.0


## 10. Conclusões e pontos em aberto

- **Deduplicação:** nenhuma duplicata encontrada nesta execução — a etapa segue como proteção estrutural para futuras cargas.
- **Nulos:** poucas linhas de `quantidade` nula descartadas (procedimentos não reportados por algum país/ano).
- **País:** `USA`/`US`, `UK`/`UNITED KINGDOM` e `TURKEY`/`TURKIYE` unificados; nenhuma outra variante restante foi identificada na EDA.
- **Tradução:** `categoria_pt` e `procedimento_pt` cobrem 100% dos valores encontrados, com o termo original em inglês preservado em `categoria`/`procedimento`.
- **Schema:** validado com sucesso — a Silver está pronta para consumo.
- **Em aberto (fora do escopo deste tratamento):** variantes de grafia do *mesmo* procedimento entre anos (`Face Lift`/`Facelift`, `Non-Surgical Fat Reduction`/`Nonsurgical Fat Reduction`, etc.) foram traduzidas para o mesmo termo em português, mas continuam como linhas/identidades separadas. Unificá-las em um único identificador de procedimento mudaria a granularidade da tabela e é uma decisão de modelagem maior — recomendada para uma futura camada Gold, se análises de série temporal por procedimento exigirem isso.